# COM2041 Week 3 Lab -- Asymmetric Cryptography
In this lab we will explore use of RSA for encryption and signature, Elgamal encryption, and Schnorr signatures

Install Pycryptodome, we will use this library to run RSA.

In [ ]:
!pip install pycryptodome

## Section 1: RSA Key generation, encryption and decryption

#### Task 1.1 
Complete the code for generating an RSA keypair, and defining encryption and decryption.

Note that pow(x,y,n) is a useful function which gives $x^y$ mod $n$.  This will be used a lot in this lab.

In [ ]:
# Task 1.1: Textbook RSA, equal-length p,q for N

from Crypto.PublicKey import RSA
from Crypto.Util.number import getPrime, inverse

def gen_rsa_key(bits_per_prime):
    """
    Generate RSA key pair.
    Uses two primes p, q of equal bit length to form N = p*q.
    Returns a RSA key object.
    """
    p = getPrime(bits_per_prime)
    q = getPrime(bits_per_prime)
    n = p * q
    e = 65537  # Common public exponent, trade-off between security and efficiency
    phi = (p - 1) * (q - 1)
    d = inverse(e, phi)
    # RSA.construct(tuple): (n, e, d, p, q)
    key = RSA.construct((n, e, d, p, q))
    return key

def textbook_rsa_encrypt(key, m_int):
    """
    Textbook RSA encryption: C = m^e mod N.
    Uses the key's public components (n, e).
    """
    return pow(m_int, key.e, key.n)


def textbook_rsa_decrypt(key, c):
    """
    Textbook RSA decryption: m = C^d mod N.
    Uses the key's private exponent d and modulus n.
    """
    return pow(c, key.d, key.n)

### Task 1.2: Generate a 512-bit RSA key
For the purposes of the lab, we first choose 256-bit prime numbers p and q to generate a 512-bit RSA key

In [ ]:
# Generate key pair (equal-length p, q via RSA object)
key1 = gen_rsa_key(256)
p, q = key1.p, key1.q
print("RSA key — bit length of primes p, q:", p.bit_length(), q.bit_length())
print("Bit length of N:", key1.n.bit_length())

### Encoding, Encrypting and Decrypting

The following code encrypts and then decrypts a plaintext.  Intermediate stages are printed showing the values as bytes, then as an integer, and then encrypting and decrypting and retrieving the original plaintext bytes.  It is initially run on "Example text".

In [ ]:
def encrypt_and_decrypt_full(key, plaintext):
    byte_data = plaintext.encode("ascii")
    print("Plaintext as bytes =", byte_data)
    print("\nPlaintext as hex =",byte_data.hex())
    value = int.from_bytes(byte_data, byteorder="big")
    print("\nPlaintext as integer =", value)
    cipher = textbook_rsa_encrypt(key, value)
    print("\nCiphertext as integer =",cipher)
    plain = textbook_rsa_decrypt(key, cipher)
    print("\nDecrypted ciphertext as integer =",plain)
    byte_data1 = plain.to_bytes((plain.bit_length() + 7) // 8, byteorder="big")
    print("\nDecrypted data as hex =",byte_data1.hex())
    print("\nDecrypted ciphertext converted back to bytes:",byte_data1)

encrypt_and_decrypt_full(key1,"Example text")

### Task 1.3 
Run encrypt_and_decrypt on plaintext1 below.  Check that the decrypted values match the values before they were encrypted.
#### Answer
The decrypted text indeed matches the original plaintext.

In [ ]:
plaintext1 = "I hear and I forget."
encrypt_and_decrypt_full(key1,plaintext1)

### Task 1.4
Now encrypt plaintext2 and then decrypt it.  Show that the resulting text is the same as the original.
#### Answer
The decrypted text indeed matches the original plaintext.

In [ ]:
plaintext2 = "I hear and I forget. I see and I remember."
print("Plaintext =", plaintext2)
cipher = textbook_rsa_encrypt(key1, int.from_bytes(plaintext2.encode("ascii"), byteorder="big"))
plain = textbook_rsa_decrypt(key1, cipher)
print("\nDecrypted ciphertext converted back to bytes:",plain.to_bytes((plain.bit_length() + 7) // 8, byteorder="big"))

### Task 1.5
Now encrypt plaintext3 and then decrypt it. What is the final decrypted plaintext?  Can you explain what has occurred?
#### Answer
The resulting plaintext is different to the original, and is garbled and an arbitrary string of bytes.

What has occurred is that plaintext3 is longer than the modulus N of the RSA key.  Given that all calculations for RSA encryption and decryption are modulo N, the final decrypted plaintext is the sequence of bytes associated with plaintext3 modulo N

In [ ]:
plaintext3 = "I hear and I forget. I see and I remember. I do and I understand. - Confucius"
print("Plaintext =", plaintext3)
cipher = textbook_rsa_encrypt(key1, int.from_bytes(plaintext3.encode("ascii"), byteorder="big"))
plain = textbook_rsa_decrypt(key1, cipher)
print("\nDecrypted ciphertext converted back to bytes:",plain.to_bytes((plain.bit_length() + 7) // 8, byteorder="big"))


### Task 1.6 - same as 1.5 but with a 1024-bit key
Generate a key with 512 bit primes.  Encrypt and then decrypt plaintext3 with your new key.  What is the result?
#### Answer
Now the value of the plaintext is less than the modulus N of the key, so the encryption and decryption of the plaintext will return the original plaintext.

In [ ]:
key2 = gen_rsa_key(512)
print("Plaintext =", plaintext3)
cipher = textbook_rsa_encrypt(key2, int.from_bytes(plaintext3.encode("ascii"), byteorder="big"))
plain = textbook_rsa_decrypt(key2, cipher)
print("\nDecrypted ciphertext converted back to bytes:",plain.to_bytes((plain.bit_length() + 7) // 8, byteorder="big"))


#### Task 1.7: Homomorphic encryption
Use key2 (Task 1.6) to illustrate the following example of homomorphic encryption with textbook RSA.  Can you explain the final result?
#### Answer
The original plaintext "ha" is two bytes, or 16 bits.  If we multiply the value of "ha" by $2^{16}$ then in bytes we end up with the bytes for "ha" followed by two bytes of value 0.  Similarly multiplying by $2^{32}$ gives the bytes for "ha" followed by 4 bytes of value 0, etc.  Multiplying by $(1 + 2^{16} + 2^{32} + 2^{48} + 2^{64})$ adds all of these together, resulting in the bytes for "hahahahaha".  The multiplication is performed within the encryption, by means of homomorphic encryption, and then the result is decrypted to obtain the resulting plaintext.

In [ ]:
# Encrypt the text "ha"
# Encrypt the integer (1 + 2**16 + 2** 32 + 2**48 + 2**64)
# Multiply the two ciphertexts (modulo N)
# Decrypt the result

enc_ha = textbook_rsa_encrypt(key2,int.from_bytes("ha".encode("ascii"), byteorder="big"))
enc_num = textbook_rsa_encrypt(key2,(1 + 2**16 + 2**32+ 2**48 + 2**64))
product = enc_ha * enc_num % key2.n
decryption = textbook_rsa_decrypt(key2, product)

print("Encryption of \"ha\" is:",enc_ha)
print("Encryption of (1 + 2**16 + 2**32 + 2**48 + 2**64) is:",enc_num)
print("Product of encryptions modulo N is:",product)
print("Decryption of product is:",decryption)
print("\nDecryption of product as ascii is:",decryption.to_bytes((decryption.bit_length() + 7) // 8, byteorder="big").decode("ascii"))

#### Task 1.8  Export a public key
Public keys are intended to be shared.  The standard format for doing this is PEM (Privacy Enhanced Mail) format.  
Use the next cell to print the PEM format for the key generated in task 1.2.  We also print the public values of e and n.


In [ ]:
pem1 = key1.public_key().export_key().decode("ascii")
print(pem1)

#### Task 1.9 Export the public key from Task 1.6
This is a larger public key, as the primes were 512 bits rather than 256 bits each.  Generate the PEM for the key.  Is it longer than the PEM from Task 1.8?

In [ ]:
# Your code here: generate the PEM format for key2
pem2 = key2.public_key().export_key().decode("ascii")
print(pem2)
print("\nThis public key corresponds to:\ne =",public_key.e)
print("n =",public_key.n)

#### Task 1.10 Import a public key
Copy the entire text of the pem you have just generated, to define input_pem.  Check that the result has the same values for e and n as the key that generated the PEM text.

In [ ]:
# using the pem format (Privacy Enhanced Mail)

input_pem = """-----BEGIN PUBLIC KEY-----
MIGfMA0GCSqGSIb3DQEBAQUAA4GNADCBiQKBgQCTJ3QVltreKvM9y7Suew2SqMk9
22r+rBAiQqF38J5oNgrhJR1z/cjR3lMiGcBXAzicAkka61o0+1mSKs8Yrevn9/x8
mx1kk3FsHkgKL08Dv9glCl9JwKPTP+gPFW11+K1dymt6A2t4JWtZW5ADY2qphyf7
RQxVshjmioLw2q5/6QIDAQAB
-----END PUBLIC KEY-----
"""

imported_public_key = RSA.import_key(input_pem.encode("ascii"))
print(imported_public_key.export_key().decode("ascii"))
print("\nThis public key corresponds to:\ne =",imported_public_key.e)
print("n =",imported_public_key.n)

## Section 2: Working with Alice's public RSA key
You are not Alice.  So you have access to her public key but you don't have her secret key.

Alice's RSA public key is given in PEM format as follows:

#### Task 2.1  Read Alice's key into a variable so you can use it

In [ ]:
input_alice_pem = """-----BEGIN PUBLIC KEY-----
MIGfMA0GCSqGSIb3DQEBAQUAA4GNADCBiQKBgQCTJ3QVltreKvM9y7Suew2SqMk9
22r+rBAiQqF38J5oNgrhJR1z/cjR3lMiGcBXAzicAkka61o0+1mSKs8Yrevn9/x8
mx1kk3FsHkgKL08Dv9glCl9JwKPTP+gPFW11+K1dymt6A2t4JWtZW5ADY2qphyf7
RQxVshjmioLw2q5/6QIDAQAB
-----END PUBLIC KEY-----
"""

Alice_public_key = RSA.import_key(input_alice_pem.encode("ascii"))

#### Task 2.2: Guess a message
Bob has sent Alice a message, encrypted with her public key using textbook RSA (defined in Task 1.1).  Mallory has intercepted the ciphertext and wants to discover what the message was by guessing.

The intercepted integer corresponding to the message is: 61639781639839351341215282422943006208714169255602354358172954253621176270250187090457532773069353004241296090202084403569625171438590593426812198747774399610506533795653262598245123128792356142688656366782974108652711642498330366923606555222594920185955602910701638582651905705960056581573254048878221712753

Mallory thinks that the message could be "Meet me at two", "Meet me at three", "Meet me at four" or "Meet me at seven".  Help Mallory find which of these guesses is correct.
#### Answer
"Meet me at four" is the correct answer, discovered by matching its encryption to the intercepted message.  Note that Alice's private key is not needed to do this, only her public key.


In [ ]:
# "Meet me at four" is the correct answer, discovered by matching the following to the intercepted message:
textbook_rsa_encrypt(Alice_public_key,int.from_bytes("Meet me at four".encode("ascii"), byteorder="big"))

# Note that you don't need Alice's secret key to check this.

#### Task 2.3: checking signatures
Complete the code for rsa_verify_with_hash(key, m_int, sig).  Encrypt **sig** with **key**.  If this matches the hash of **m_int** then the signature is correct, otherwise it does not verify.  Your code shoudl verify the signature in the following cell.

In [ ]:
import hashlib

def hash_sha256(m_int):
    """Hash message (int) to a fixed-length value; used in verify."""
    h = hashlib.sha256(m_int.to_bytes((m_int.bit_length() + 7) // 8, "big")).digest()
    return int.from_bytes(h, "big")

def rsa_verify_with_hash(key, m_int, sig):
    """Verify with hash: E(key,sig) ≡ H(m)"""
    hm = hash_sha256(m_int)
    check = textbook_rsa_encrypt(key, sig)
    if check == hm:
        return "The signature verification is correct"
    else:
        return "The signature does not verify"

The following should verify a correct signature:

In [ ]:
sig = 71395072891092459417471129433928793652549974140686975803983245204224810808442697988574953801839699584283475192057066331482898380030329730745133328496574447711036202153069715442511509517614598613060653268583852461389141385868158566948706592150892014572677229789777629581754978999745806123127389316196886768442
rsa_verify_with_hash(Alice_public_key, 97, sig)

#### Task 2.4: verifying a message from Alice
Bob has received four messages that are apparently from Alice.  However only one signature has arrived.  Can you tell which of these messages is genuinely from Alice?
#### Answer
The correct message is identified in the code below.

In [ ]:
message1 = "I'm busy today"
message2 = "I'm not free now but will be in touch later"
message3 = "How about meeting at 11?"
message4 = "I think someone is pretending to be me"

signature = 74419248893668972410495485032988920251637366854159892657809445605028299164895310949222379682707278996147238595192379728968161101881669448787194986073874591396689612672085959638386953906208763736150020024374630163933228270442936845517556912984860144380387469544210415972761160082612483153083173828130768355307

print(rsa_verify_with_hash(Alice_public_key, int.from_bytes("How about meeting at 11?".encode("ascii"), byteorder="big") , signature))

#### Task 2.5 - RSA with OAEP
Create an RSA-with-OAEP cipher, using Alice's Public Key.  This will apply OAEP randomised padding before applying the RSA encryption.
Select a plaintext and create two encryptions of same plaintext, using the same cipher.  Check whether the two ciphertexts are the same or different.  What does this mean for a guessing attack?

In [ ]:
from Crypto.Cipher import PKCS1_OAEP
secure_cipher = PKCS1_OAEP.new(Alice_public_key)
ciphertext = secure_cipher.encrypt(b"Secret message").hex()
ciphertext2 = secure_cipher.encrypt(b"Secret message").hex()
print("First encryption of \"Secret message\" is",ciphertext)
print("\nSecond encryption of \"Secret message\" is",ciphertext2)

## Section 3 Elgamal

#### Task 3.1  Complete the code
Firstly we provide methods for generating Elgamal keys, and for encrypting and decrypting.  

In [ ]:
from Crypto.Util.number import getPrime, inverse
from Crypto.Random import get_random_bytes

def elgamal_keygen(bits):
    """ElGamal encryption key: p prime, g generator, x secret, y = g^x mod p."""
    p = getPrime(bits)
    g = 2
    if pow(g, (p - 1) // 2, p) == 1:
        g = 3
    x = int.from_bytes(get_random_bytes((bits + 7) // 8), "big") % (p - 1)
    if x == 0:
        x = 1
    y = pow(g, x, p)
    return (p, g, y), (p, g, x)

def elgamal_encrypt(pk, m_int):
    """Encrypt message m: (c1, c2) = (g^k, m * y^k) mod p."""
    p, g, y = pk
    r = int.from_bytes(get_random_bytes((p.bit_length() + 7) // 8), "big") % (p - 1)
    if r == 0:
        r = 1
    c1 = pow(g, r, p)
    c2 = (m_int * pow(y, r, p)) % p
    return (c1, c2)

def elgamal_decrypt(sk, c):
    """Decrypt: m = c2 * (c1^x)^(-1) mod p."""
    p, g, x = sk
    c1, c2 = c
    s = pow(c1, x, p)
    return (c2 * inverse(s, p)) % p

#### Task 3.2 - Preventing guessing attacks
In this task we show that different encryptions of the same plaintext give different ciphertexts.  Here we generate a key, and then ecrpyt the same plaintext twice, and obtain different ciphertexts for the two encryptions:

In [ ]:
eg_public, eg_secret = elgamal_keygen(512)

In [ ]:
enc1 = elgamal_encrypt(eg_public,3)
enc2 = elgamal_encrypt(eg_public,3)
print("enc1 =",enc1)
print("enc2 =",enc2)

Now decrypt both ciphertexts.  Observe that the same (correct) plaintext is retrieved in both cases.

In [ ]:
dec1 = elgamal_decrypt(eg_secret,enc1)
dec2 = elgamal_decrypt(eg_secret,enc2)
print("dec1 =",dec1)
print("dec2 =",dec2)

#### Task 3.3 Elgamal homomorphic encryption
We can use homomorphic encryption to combine two ciphertexts each encrypting a value so that the result encrypts the product of the two original values.  Give the code for defining **enc5** in terms of **enc3** and **enc4** so that it encrypts the product.

In [ ]:
p = eg_public[0]  # this is the prime for the elgamal cipher
value3 = 13
value4 = 11
enc3 = elgamal_encrypt(eg_public,value3)
enc4 = elgamal_encrypt(eg_public,value4)
enc5 = (enc3[0] * enc4[0] % p, enc3[1] * enc4[1] % p)
dec5 = elgamal_decrypt(eg_secret,enc5)
print("dec5 =",dec5)

## Section 4: Schnorr Signature

The Schnorr signature scheme is efficient and secure. It is defined as follows:

**Scheme:**  
- **Keygen:** Prime $q$, generator $g \in 1..(q-1)$, secret key is a value $x$, public key is $y = g^{-x} \bmod q$.  

- **Sign:** To generate a signature on message $M$: Choose randomness $r$ between $2$ and $q-1$.  Compute $c$ and $s$ as follows: $c = hash(g^r \| M) \bmod q$; $s = r + c.x \bmod q.$ The signature is the pair $(c, s)$.

- **Verify:** Check whether $c \stackrel{?}{=} hash((g^s \cdot y^{c}) \bmod q \| M)$. In other words, compute $hash ((g^s \cdot y^{c} \bmod q) \| M)$ and check if it is equal to $c$. 

#### Task 4.1
Provide the definition of **schnorr_verify** to complete the Schnorr signature code below which implements the Schnorr signature scheme.

In [ ]:
# Task 3: Schnorr signature (keygen, sign, verify)
from Crypto.Random import get_random_bytes
import hashlib
from Crypto.Util.number import getPrime, inverse, isPrime

def schnorr_keygen(q_bits):
    """Schnorr keygen: q prime, g generator, secret x, public y = g^a mod p."""
    q = getPrime(q_bits)
    # g = 2^((p-1)/q) mod p has order q
    g = 2
    if pow(g, (q - 1) // 2, q) == 1:
        g = 3
    x = int.from_bytes(get_random_bytes(32), "big") % q
    if x == 0 or x == 1: # highly unlikely but just in case, since x=0 doesn't work
        x = 2
    minusx = 0-x
    y = pow(g,minusx, q)
    return (q, g, y), (q, g, x)

def schnorr_concat_and_hash(r: int, m: bytes, q: int) -> int:
    """Challenge e = H(r || m) mod q."""
    h = hashlib.sha256(r.to_bytes((r.bit_length() + 7) // 8 or 1, "big") + m).digest()
    return int.from_bytes(h, "big") % q

def schnorr_sign(sk, m: bytes):
    """Sign: c = hash(g^r||m), s = r + c*x mod (q-1); return (c, s)."""
    q, g, x = sk
    r = int.from_bytes(get_random_bytes(32), "big") % q
    if r == 0 or r == 1:
        r = 2
    rexp = pow(g, r, q)
    c = schnorr_concat_and_hash(rexp, m, q)
    s = (r + c * x) % (q-1)
    return (c, s)

def schnorr_verify(pk, m: bytes, sig):
    """Verify: c == hash(g^s * y^c mod q || M)."""
    q, g, y = pk
    c, s = sig
    v = (pow(g, s, q) * pow(y, c, q)) % q
    c2 = schnorr_concat_and_hash(v, m, q)
    return c == c2

#### Task 4.2
Test your code by checking the following two signatures.  One should succeed and one should fail the signature check

In [ ]:
alice_verify_key = (100127909518703182357267516905029098820866205400436620572006420598492841263959, 3, 17430792015334668148991866148421143841235969815411887801248421265675454506440)
msg1 = b"hello schnorr"
sig1 = (41073702261550845348090789224754795509272852242117143141011644035676266482480, 78196967169734129093953299034314501807170381122240174481169976021090797941668)
print("Schnorr verify msg1:", schnorr_verify(alice_verify_key, msg1, sig1))

bob_verify_key = (60002312796942155940331108762748220032041666727276189299521322204727418477109, 2, 42640774353831605271770311750589727251408321815264209692945255415551255114655)
msg2 = b"good morning elgamal"
sig2 = (41430115786214947544682545006508786681065484653767348954310467189808032627661, 10788634731987573245678473868879199942251248588083204572900040011421307825735)
print("Schnorr verify msg2:", schnorr_verify(bob_verify_key, msg2, sig2))


#### Task 4.3 Match the signatures to the message
We have a selection of verify keys, messages, and signatures. Everyone has signed a message. Unfortunately the signatures have become detatched from the keys and the messages.  Can you tell who has signed which messages?
#### Answer: 
The schnorr_verify matches provided below show who has signed which message.  Note that both sandy and charlie have signed msg1, illustrating that a message can be signed by anyone.  Nobody has signed msg4.

In [ ]:
msg1 = b"Life is like a box of chocolates. You never know what you're gonna get."
msg2 = b'Genius is one percent inspiration and ninety-nine percent perspiration.'
msg3 = b"The best way to predict the future is to create it."
msg4 = b"Be yourself; everyone else is already taken."

charlie_verify_key = (14486560890333321061, 2, 10023042836943747717)
chris_verify_key = (18307287252547363579, 2, 7720157265561343247)
jerry_verify_key = (10792935745120020887, 3, 3247374053036510948)
sandy_verify_key = (9891175841028161257, 3, 3537215357179095687)

sigA = (5891021249944993668, 1067199103268824346)
sigB = (5988921025064014521, 629931455232909358)
sigC = (4172389596059496293, 7749467107204080996)
sigD = (11546161560214415835, 10478488453985900116)

# Answer which confirms the matches of the verify keys, messages, and signatures
print("Schnorr verify:", schnorr_verify(chris_verify_key, msg3, sigA))
print("Schnorr verify:", schnorr_verify(jerry_verify_key, msg2, sigB))
print("Schnorr verify:", schnorr_verify(sandy_verify_key, msg1, sigC))
print("Schnorr verify:", schnorr_verify(charlie_verify_key, msg1, sigD))